# 第115章 模型保存与批量推理

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 30 / 34 步：调优、比较、解释并保存模型**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 特征选择与模型解释  →  **本章任务：** 模型保存与批量推理  →  **下一步：** 用户消费价值预测项目
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

训练一个模型往往很费力，但真正能反复使用的，是把模型连同预处理一起保存下来。


## 本章目标

学完本章，你将能够：

- **理解**：理解「模型保存与批量推理」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「模型保存与批量推理」的关键输出指标。
- **迁移**：能把「模型保存与批量推理」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：训练一个模型往往很费力，但真正能反复使用的，是把模型连同预处理一起保存下来。你不太可能每次都用几百行训练代码重跑一遍，而是希望把训练好的成果存成一个文件，需要时随时加载并对新数据批量给出预测。这就要求我们想清楚保存什么、怎么校验输入，以及为什么不能从不可信来源加载。掌握了这些，你就能把训练阶段与上线阶段真正分开。

- 模型制品必须包含预处理（打个比方：要存档就存“教练+他那套训练方法”；只存教练却忘了记方法，新数据一来就得重新讲一遍，还可能跑偏。）
- 推理输入契约包括字段、类型和单位
- 序列化文件不能从不可信来源加载
- 版本、训练时间和特征清单应随制品记录


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `pickle.dumps()`、`pickle.loads()`、`model.feature_names_in_.tolist()`、`.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 只保存估计器而漏掉预处理 |
| 模型、公式与诊断 | `X.head()`、`restored.feature_names_in_.tolist()`、`batch.columns.tolist()`、`batch.assign()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 不校验输入字段和单位 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-115 -->
### 数学推导｜批量推理的阈值合同

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜完整 Pipeline 输出概率。** $p_i=f_{version}(x_i)$。

**第 2 步｜固定已审批阈值形成行动标记。** $a_i=\mathbf1(p_i\ge t)$。

**第 3 步｜预估批次负载。** 

$$
N_{action}=\sum_ia_i,
\qquad
action\ rate=\frac{N_{action}}{n}
$$

概率、阈值、模型版本和特征版本共同构成可复现的推理合同。

**把上面的关系收束为本章计算式：**

$$
\hat{y}_i=\mathbf{1}(p_i\ge t),\qquad action\ rate=\frac{1}{n}\sum_i\hat{y}_i
$$

**符号解释：** $t$ 是部署阈值，action rate 是被模型触发行动的样本比例。

**代码对应：** 保存完整 Pipeline，并在批量预测结果中同时输出概率、阈值和版本。

**使用边界：** 只保存模型主体会丢失预处理；特征缺失或顺序变化必须阻止推理。


In [ ]:
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pickle

data = load_iris(as_frame=True)
X, y = data.data, data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)).fit(
    X, y
)
payload = pickle.dumps(model)
restored = pickle.loads(payload)
print("制品字节数:", len(payload), "特征:", model.feature_names_in_.tolist())


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：模型保存后的第一件事就是对“新数据”做输入校验。请复制 `X` 的前 2 行作为一批新数据，把它的列按模型记录的 `feature_names_in_` 顺序排好，再列出结果。试着只修改一个数据字段——比如把某列列名改掉一个字——观察校验与预测会如何变化，这能帮你理解“推理输入契约”。把下方代码里的空格填上并运行，核对自检是否通过。


In [ ]:
try:
    pass
    # 请在下方填写代码
    # 目标：对 X 的前 2 行做列顺序对齐，并完成批量预测。
    # 提示：可先打印 X.columns.tolist() 与 restored.feature_names_in_.tolist() 对比。
    # 填空 ①：取模型记录的特征顺序，并据此给 batch_new 排好列
    # 填空 ②：核对列名与顺序是否一致（得到一个布尔值）
    # 填空 ③：仅当列名与顺序一致时才预测，否则置为 None

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd
import numpy as np

batch = X.head(5).copy()
required = restored.feature_names_in_.tolist()
_check_1 = bool(batch.columns.tolist() == required)
print(
    "自检 1：batch.columns.tolist()==required ->",
    "通过" if _check_1 else "需要检查",
)
if not _check_1:
    print("建议：", "请回看输入、处理步骤和预期结果。")
output = batch.assign(
    prediction=restored.predict(batch),
    confidence=restored.predict_proba(batch).max(axis=1),
)
display(output)
print(
    "重载预测一致:",
    np.array_equal(model.predict(batch), restored.predict(batch)),
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 只保存估计器而漏掉预处理
- 不校验输入字段和单位
- 加载不可信 pickle
- 没有记录 sklearn 版本和训练数据版本


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 115.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 115.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 115.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

把预处理与模型作为单一制品保存，完成输入校验、批量推理和结果复核。


### 你已经掌握

- 保存完整 Pipeline
- 加载模型并复现预测
- 校验列名与顺序
- 设计批量预测输出


### 需要注意

- 只保存估计器而漏掉预处理
- 不校验输入字段和单位
- 加载不可信 pickle
- 没有记录 sklearn 版本和训练数据版本


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案
batch_new = X.head(2).copy()
feature_order = restored.feature_names_in_.tolist()
batch_new = batch_new[feature_order]
columns_ok = bool(batch_new.columns.tolist() == feature_order)
predictions = restored.predict(batch_new) if columns_ok else None


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_batch = X.tail(3).copy()
practice_output = pd.DataFrame(
    {
        "prediction": restored.predict(practice_batch),
        "confidence": restored.predict_proba(practice_batch).max(axis=1),
    }
)
display(practice_output)
